# Practical 02 — Symbolic Manipulation and Pattern Matching

**Artificial Intelligence 3(2-1) · BS Computer Science · Semester III**

| | |
|---|---|
| **Week** | 2 |
| **Related lectures** | Lectures 4, 18 and 19 |
| **Duration** | 3 hours |
| **Submission** | This notebook, run top to bottom from a clean kernel |

## 1. Objectives

By the end of this session you will be able to:

1. Represent statements as structured data rather than as plain strings.
2. Write a pattern matcher that recognises a template containing variables.
3. Build a transformation system that rewrites one expression into another.
4. Construct a conversational responder using pattern matching alone.
5. Explain what your program does and does not represent.

## 2. Background theory

Lecture 4 made the point that a machine manipulates symbols without any access to what they mean, and Lecture 19 showed two historical programs that exploited exactly this — producing responses that convinced people they were understood, using nothing but pattern matching and rearrangement.

Today you build the same machinery. A **pattern** is a template containing variables; **matching** is finding values for those variables that make the pattern identical to the input; and a **transformation** rewrites the captured pieces into a new expression. This is the informal ancestor of the unification procedure you will formalise in Practical 3.

**Key points to keep in mind while you work:**

- A pattern with a variable matches many inputs; the match returns what the variable captured.
- The program has no notion of what any symbol refers to — only its shape.
- Swapping person words ('my' to 'your') is what makes a reply read as though it engaged.
- If a fluent reply convinces a person, that says something about people, not about the program.

## 3. Apparatus and setup

**Software:** Python 3.10 or later. Standard library only — no external packages needed.

Nothing to install. Run the cell to confirm your kernel is working.

In [1]:
print('Ready. No external packages required for this practical.')

Ready. No external packages required for this practical.


## 4. Procedure

Work through these steps in order. The code here is given to you and should run as it stands — read it, run it, and make sure you understand what each part does before moving on to your own tasks in Section 5.

### Step 1 — Representing a statement as structure, not text

A string is opaque: you cannot easily ask what its parts are. Representing a statement as a list of tokens lets you inspect and rearrange it.

In [2]:
sentence = 'i am worried about my exams'
tokens = sentence.split()
print('as a string :', repr(sentence))
print('as tokens   :', tokens)
print('third token :', tokens[2])

as a string : 'i am worried about my exams'
as tokens   : ['i', 'am', 'worried', 'about', 'my', 'exams']
third token : worried


### Step 2 — A matcher for patterns with one variable

The pattern below uses `?X` to mean 'capture everything here'. The matcher returns a dictionary of what each variable captured, or None if the pattern does not fit.

In [3]:
def match(pattern, tokens):
    """Match a token list against a pattern. '?X' captures the rest.
    Returns a dict of bindings, or None if there is no match."""
    bindings = {}
    for i, p in enumerate(pattern):
        if p.startswith('?'):
            bindings[p] = tokens[i:]      # capture everything remaining
            return bindings
        if i >= len(tokens) or p != tokens[i]:
            return None                   # a constant word did not match
    return bindings if len(pattern) == len(tokens) else None

print(match(['i', 'am', '?X'], 'i am worried about my exams'.split()))
print(match(['i', 'am', '?X'], 'she is worried'.split()))

{'?X': ['worried', 'about', 'my', 'exams']}
None


### Step 3 — Transforming what was captured

Once you have captured a phrase, you can rewrite it. Swapping first-person for second-person words is what makes the reply read naturally.

In [4]:
SWAP = {'my': 'your', 'i': 'you', 'me': 'you', 'am': 'are', 'mine': 'yours'}

def swap_person(tokens):
    return [SWAP.get(t, t) for t in tokens]

captured = 'worried about my exams'.split()
print('captured:', captured)
print('swapped :', swap_person(captured))

captured: ['worried', 'about', 'my', 'exams']
swapped : ['worried', 'about', 'your', 'exams']


### Step 4 — Putting it together into a responder

A rule is a pattern plus a reply template. The responder tries each rule in turn and falls back on a generic prompt when nothing matches.

In [5]:
RULES = [
    (['i', 'am', '?X'],      'How long have you been {X}?'),
    (['i', 'need', '?X'],    'Why do you need {X}?'),
    (['i', 'cannot', '?X'],  'What stops you being able to {X}?'),
]
FALLBACK = 'Tell me more about that.'

def respond(sentence):
    tokens = sentence.lower().split()
    for pattern, template in RULES:
        b = match(pattern, tokens)
        if b is not None:
            captured = swap_person(b['?X'])
            return template.format(X=' '.join(captured))
    return FALLBACK

for s in ['I am worried about my exams',
          'I need more time',
          'The weather is fine today']:
    print('%-32s -> %s' % (s, respond(s)))

I am worried about my exams      -> How long have you been worried about your exams?
I need more time                 -> Why do you need more time?
The weather is fine today        -> Tell me more about that.


## 5. Tasks

These are your work. Each cell marked `# TODO` is for you to complete. Write your own code; do not copy the procedure cells unchanged.

### Task 1 — Add five rules of your own

Extend the RULES table with at least five new patterns. At least two of them must capture a phrase and reuse it in the reply.

> **Marks:** 4

In [6]:
import re

def tokenise(sentence):
    """Lower-case the sentence and drop punctuation, so 'Hello!' == 'hello'."""
    return re.findall(r"[a-z']+", sentence.lower())

def respond(sentence, rules=None, matcher=match):
    """Try every rule in order; the first pattern that matches wins."""
    rules = RULES if rules is None else rules
    tokens = tokenise(sentence)
    for pattern, template in rules:
        b = matcher(pattern, tokens)
        if b is not None:
            # swap person words in every captured phrase, then fill the template
            filled = {k[1:]: ' '.join(swap_person(v)) for k, v in b.items()}
            return template.format(**filled)
    return FALLBACK

# five NEW rules (feel, want and my capture a phrase; hello and because do not)
NEW_RULES = [
    (['i', 'feel', '?X'],   'Why do you feel {X}?'),
    (['i', 'want', '?X'],   'Why do you want {X}?'),
    (['my', '?X'],          'Tell me more about your {X}.'),
    (['hello'],             'Hello! How are you feeling today?'),
    (['because', '?X'],     'Is that the real reason?'),
]
RULES_EXT = RULES + NEW_RULES

tests = [
    'I feel very tired',            # rule: i feel ?X
    'I want to leave my job',       # rule: i want ?X
    'My sister never calls me',     # rule: my ?X
    'Hello!',                       # rule: hello
    'Because I failed the test',    # rule: because ?X
    'I am worried about my exams',  # an old rule still works
    'The weather is fine today',    # nothing matches -> fallback
]
for s in tests:
    print('%-30s -> %s' % (s, respond(s, RULES_EXT)))

I feel very tired              -> Why do you feel very tired?
I want to leave my job         -> Why do you want to leave your job?
My sister never calls me       -> Tell me more about your sister never calls you.
Hello!                         -> Hello! How are you feeling today?
Because I failed the test      -> Is that the real reason?
I am worried about my exams    -> How long have you been worried about your exams?
The weather is fine today      -> Tell me more about that.


### Task 2 — Handle a variable in the middle of a pattern

The matcher above only captures at the end of a pattern. Extend it so a pattern such as `['i', '?X', 'about', 'exams']` works — the variable captures the words between two fixed anchors.

> **Marks:** 5

In [7]:
def match2(pattern, tokens, bindings=None):
    """Improved matcher: a '?X' variable can appear ANYWHERE in the pattern
    (start, middle or end) and there can be several variables.
    A variable captures one or more words. Uses recursion + backtracking:
    try a short capture first, and if the rest of the pattern fails,
    try a longer capture.
    Returns a dict of bindings, or None if there is no match."""
    if bindings is None:
        bindings = {}
    if not pattern:                                   # pattern used up ...
        return bindings if not tokens else None       # ... so tokens must be used up too
    p = pattern[0]
    if p.startswith('?'):                             # a variable
        for end in range(1, len(tokens) + 1):         # capture tokens[:end]
            captured = tokens[:end]
            if p in bindings and bindings[p] != captured:
                continue                              # same variable must capture the same words
            new_b = dict(bindings)
            new_b[p] = captured
            result = match2(pattern[1:], tokens[end:], new_b)
            if result is not None:
                return result
        return None
    if tokens and tokens[0] == p:                     # a constant word must match exactly
        return match2(pattern[1:], tokens[1:], bindings)
    return None

# --- tests ---
print(match2(['i', '?X', 'about', 'exams'], 'i worry about exams'.split()))
print(match2(['i', '?X', 'about', 'exams'], 'i am worried about exams'.split()))
print(match2(['i', '?X', 'about', '?Y'],    'i am worried about my exams'.split()))
print(match2(['?X', 'is', 'fun'],           'python is fun'.split()))
print(match2(['i', '?X', 'about', 'exams'], 'i am worried about sports'.split()))   # no match -> None
print(match2(['i', 'am', '?X'],             'she is worried'.split()))              # no match -> None

# --- use it inside the responder with a middle-variable rule ---
MIDDLE_RULES = [(['i', '?X', 'about', '?Y'], 'You said you {X} about {Y}. Tell me more.')] + RULES_EXT
print()
for s in ['I am worried about my exams', 'I worry about exams', 'I need more time']:
    print('%-30s -> %s' % (s, respond(s, MIDDLE_RULES, matcher=match2)))

{'?X': ['worry']}
{'?X': ['am', 'worried']}
{'?X': ['am', 'worried'], '?Y': ['my', 'exams']}
{'?X': ['python']}
None
None

I am worried about my exams    -> You said you are worried about your exams. Tell me more.
I worry about exams            -> You said you worry about exams. Tell me more.
I need more time               -> Why do you need more time?


### Task 3 — Build a simple expression simplifier

Lecture 20 described simplification by rewrite rules. Represent the expression `1 * x + 0` as a nested list and write rules that reduce it to `x`. Apply the rules repeatedly until nothing changes.

> **Marks:** 5

In [8]:
# An expression is a nested list: [operator, left, right].
# 1 * x + 0   is written as   ['+', ['*', 1, 'x'], 0]
expr = ['+', ['*', 1, 'x'], 0]

# Rewrite rules: (pattern, replacement). '?A' is a variable, everything else is a constant.
ALGEBRA_RULES = [
    (['+', '?A', 0],   '?A'),     # a + 0 -> a
    (['+', 0, '?A'],   '?A'),     # 0 + a -> a
    (['-', '?A', 0],   '?A'),     # a - 0 -> a
    (['*', 1, '?A'],   '?A'),     # 1 * a -> a
    (['*', '?A', 1],   '?A'),     # a * 1 -> a
    (['*', 0, '?A'],   0),        # 0 * a -> 0
    (['*', '?A', 0],   0),        # a * 0 -> 0
    (['^', '?A', 1],   '?A'),     # a ^ 1 -> a
    (['^', '?A', 0],   1),        # a ^ 0 -> 1
]

def is_var(x):
    return isinstance(x, str) and x.startswith('?')

def match_expr(p, e, b):
    """Match pattern p against expression e (both may be nested lists)."""
    if is_var(p):
        if p in b:
            return b if b[p] == e else None
        nb = dict(b); nb[p] = e
        return nb
    if isinstance(p, list):
        if not isinstance(e, list) or len(p) != len(e):
            return None
        for pi, ei in zip(p, e):
            b = match_expr(pi, ei, b)
            if b is None:
                return None
        return b
    return b if p == e else None

def substitute(t, b):
    """Fill the captured values into the replacement."""
    if is_var(t):
        return b[t]
    if isinstance(t, list):
        return [substitute(x, b) for x in t]
    return t

def rewrite_once(e):
    """One pass: simplify the sub-expressions first, then try the rules on the top."""
    if isinstance(e, list):
        e = [e[0]] + [rewrite_once(c) for c in e[1:]]
    for pattern, result in ALGEBRA_RULES:
        b = match_expr(pattern, e, {})
        if b is not None:
            return substitute(result, b)
    return e

def simplify(e, show=True):
    """Apply the rules repeatedly until nothing changes."""
    step = 0
    if show: print('start  :', e)
    while True:
        new = rewrite_once(e)
        if new == e:
            if show: print('stable :', e)
            return e
        step += 1
        if show: print('pass %d :' % step, new)
        e = new

print(simplify(expr))                                                     # 1*x + 0 -> x
print()
print(simplify(['*', ['+', 0, ['*', 1, 'y']], 1]))                        # (0 + 1*y) * 1 -> y
print()
print(simplify(['+', ['*', 0, 'z'], ['*', 1, ['+', 'a', 0]]]))            # 0*z + 1*(a+0) -> a
print()
print(simplify(['+', 'x', 'y']))                                          # nothing to simplify

start  : ['+', ['*', 1, 'x'], 0]
pass 1 : x
stable : x
x

start  : ['*', ['+', 0, ['*', 1, 'y']], 1]
pass 1 : y
stable : y
y

start  : ['+', ['*', 0, 'z'], ['*', 1, ['+', 'a', 0]]]
pass 1 : a
stable : a
a

start  : ['+', 'x', 'y']
stable : ['+', 'x', 'y']
['+', 'x', 'y']


### Task 4 — Test it on a classmate

Have someone else use your responder for a few exchanges without telling them how it works. Record what they assumed the program knew, in the observations table.

> **Marks:** 2

In [9]:
# No code is needed for the marks - but this small helper lets a classmate
# chat with your responder. Run chat(), let them type, and type 'quit' to stop.
def chat():
    print("Bot: Hello. What is on your mind? (type 'quit' to stop)")
    while True:
        text = input('You: ')
        if text.strip().lower() == 'quit':
            print('Bot: Goodbye.')
            break
        print('Bot:', respond(text, MIDDLE_RULES, matcher=match2))

# chat()   # <- remove the '#' to run it, then record your classmate's reaction in Section 6

### Task 5 — State what your program represents

In a markdown cell, list exactly what your program stores. Then list what a user typically assumes it stores. Be specific and honest about the gap.

> **Marks:** 4

### Task 5 — What my program stores vs. what users assume

**What my program actually stores**

- A table (`RULES`) of token-list patterns, each paired with a reply template string.
- A small dictionary (`SWAP`) that maps words like *my* to *your*.
- A fixed fallback sentence (`FALLBACK`).
- For the duration of one call, the words captured by variables (e.g. `?X = ['worried', 'about', 'my', 'exams']`) — and these are thrown away as soon as the reply is printed.
- For the simplifier: a table of rewrite rules and the current nested-list expression.

**What a user typically assumes it stores**

- That it knows what "worried" or "exams" mean, and that exams cause stress.
- That it remembers the earlier part of the conversation.
- That it has a model of the user (their feelings, their situation).
- That it *chose* a question because it understood the problem, and that it cares.

**The gap**

The program only compares the *shape* of a word list to a template. It has no meaning for any word, no memory between turns, and no model of the person. The impression of understanding comes from the human reader, helped by the person-swap (*my* → *your*), which makes the echoed words sound like a personal reply. Sentences that do not fit a rule (e.g. "The weather is fine today") immediately expose this by getting the generic fallback.

## 6. Observations

Complete this table from your own results. Do not leave it blank — the observations carry marks and are what the viva is based on.

| Input sentence | Rule that matched | Reply produced | Sensible? |
|---|---|---|---|
| I am worried about my exams | `['i', 'am', '?X']` | How long have you been worried about your exams? | Yes — the person swap makes it read naturally |
| I feel very tired | `['i', 'feel', '?X']` | Why do you feel very tired? | Yes |
| My sister never calls me | `['my', '?X']` | Tell me more about your sister never calls you. | Partly — the captured phrase is a whole clause, so the grammar breaks |
| The weather is fine today | none (fallback) | Tell me more about that. | Generic — the program has nothing to say, but it still sounds polite |

**Tester's reaction (Task 4)**

*(Draft — replace with what your classmate actually said and assumed after you test it.)*

| What they assumed the program knew | What it actually stores |
|---|---|
| That it understood they were stressed about exams | Only a pattern `['i','am','?X']` and a reply template |
| That it remembered what they said earlier | Nothing — every sentence is handled on its own |
| That it was "listening" to them | A list of tokens compared against a rule table |

## 7. Discussion

Answer in your own words, in the cell below. Two or three sentences each is enough.

1. Your responder produced sensible replies. What, if anything, does it understand? Justify your answer by reference to what is actually stored.
2. Why does swapping person words make such a large difference to how the reply reads?
3. Lecture 19 noted that people who knew how such a program worked still felt understood by it. What does that tell you about evaluating conversational systems today?

*Your answers:*

1. **What does it understand?** Nothing. The program stores only patterns, reply templates and a word-swap table. When it says "How long have you been worried about your exams?", it has just copied the captured words into a template. It has no meaning attached to "worried" or "exams" and cannot answer a follow-up like "why do exams cause stress?". Sensible-looking output does not mean understanding.

2. **Why does swapping person words matter?** Without it the reply would be "How long have you been worried about my exams?", which is clearly a machine echoing text. Turning *my* into *your* and *am* into *are* makes the reply look like it comes from a second person who is engaging with the speaker. It is a tiny rule, but it changes the reader's impression a lot.

3. **What does this tell us about evaluating conversational systems?** People felt understood even when they knew how the program worked, so how convincing a reply sounds is not a reliable test of intelligence. Today's chatbots are far more fluent, so we must be even more careful: a system should be judged by what it can actually do and represent (consistency, correctness, memory, reasoning), not only by how human its replies sound.

## 8. Viva questions

Be ready to answer these on the spot. They test whether you understand what you ran, not whether you can recite the code.

1. Show me where in your code the program decides what a sentence means. (There is no such place — can you say why?)
2. What does your matcher return when the pattern fits, and what when it does not?
3. Why did we represent the sentence as tokens rather than as a string?
4. What happens to your responder when no rule matches, and is that a good design?
5. How is your matcher related to the unification you will write next week?
6. Your tester assumed the program knew something. What, and why did they assume it?

## 9. Submission checklist

- [ ] Notebook runs top to bottom from a clean kernel with no errors
- [ ] Every `# TODO` cell is completed with your own code
- [ ] The observations table is filled in with your actual results
- [ ] Discussion answers are written in your own words
- [ ] Your name and roll number are in the cell below
- [ ] File named `P02_<your-roll-number>.ipynb`

In [10]:
NAME = ""
ROLL_NUMBER = ""
print(NAME, ROLL_NUMBER)